# Assignment 6 – Research-Oriented Phishing URL Detection

This notebook reproduces the Assignment 6 baseline and improvement experiments used on the portfolio website.

**Research question:** Can an expanded engineered feature set and class balancing improve a simple Logistic Regression baseline for phishing URL detection, and how does a balanced Random Forest compare under the same test split?


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)

df = pd.read_csv("phishing_features.csv")
df["tld"] = df["tld"].fillna("Unknown")
print(df.shape)
print(df["label"].value_counts())


## Experimental design

- Same stratified 80/20 split is used for every model.
- **Baseline:** Logistic Regression using only `url_length`, `num_dots`, and `entropy`, with no class weighting.
- **Improved Logistic Regression:** expanded engineered feature set, one-hot encoded `tld`, standardized numeric features, and `class_weight="balanced"`.
- **Improved Random Forest:** expanded engineered feature set, one-hot encoded `tld`, and class weighting.
- **Ablation test:** balanced Logistic Regression without `num_subdirs` and `tld` to test dependence on especially strong dataset signals.


In [ ]:
train_idx, test_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

train = df.iloc[train_idx].copy()
test = df.iloc[test_idx].copy()
y_train = train["label"]
y_test = test["label"]

baseline_features = ["url_length", "num_dots", "entropy"]

numeric_features = [
    "url_length", "num_dots", "has_https", "has_ip", "num_subdirs",
    "num_params", "suspicious_words", "special_char_count",
    "digits_count", "entropy"
]
categorical_features = ["tld"]


In [ ]:
baseline = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1500, random_state=42))
])

baseline.fit(train[baseline_features], y_train)


In [ ]:
preprocessor_lr = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

improved_lr = Pipeline([
    ("preprocessor", preprocessor_lr),
    ("classifier", LogisticRegression(
        max_iter=2500,
        random_state=42,
        class_weight="balanced"
    ))
])

improved_lr.fit(train[numeric_features + categorical_features], y_train)


In [ ]:
preprocessor_rf = ColumnTransformer([
    ("num", "passthrough", numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
])

improved_rf = Pipeline([
    ("preprocessor", preprocessor_rf),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

improved_rf.fit(train[numeric_features + categorical_features], y_train)


In [ ]:
ablation_features = [c for c in numeric_features if c != "num_subdirs"]

ablation_lr = Pipeline([
    ("scale", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2500,
        random_state=42,
        class_weight="balanced"
    ))
])

ablation_lr.fit(train[ablation_features], y_train)


In [ ]:
def evaluate(name, model, X):
    prediction = model.predict(X)
    probability = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, prediction, labels=[0, 1]).ravel()

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction),
        "Recall": recall_score(y_test, prediction),
        "F1": f1_score(y_test, prediction),
        "ROC_AUC": roc_auc_score(y_test, probability),
        "PR_AUC": average_precision_score(y_test, probability),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp),
    }

results = pd.DataFrame([
    evaluate("Baseline Logistic Regression", baseline, test[baseline_features]),
    evaluate(
        "Improved Balanced Logistic Regression",
        improved_lr,
        test[numeric_features + categorical_features]
    ),
    evaluate(
        "Improved Balanced Random Forest",
        improved_rf,
        test[numeric_features + categorical_features]
    ),
    evaluate(
        "Ablation: no num_subdirs or TLD",
        ablation_lr,
        test[ablation_features]
    )
])

results


## Interpretation

The baseline has high overall accuracy because phishing dominates the dataset, but it incorrectly flags a large share of legitimate URLs. The improved methods eliminate those test-set errors under this split. The result must be interpreted cautiously because the dataset contains unusually strong signals, including `num_subdirs` and TLD/missing-TLD patterns. The ablation test helps show that performance depends heavily on the available engineered features.
